In [ ]:
r"""
# 模块 09：特殊事件驱动与低风险套利 · 第 3 课：指数成分股定期调整与被动资金配置套利

> 状态：**学习中（待提交）**  
> 本课程采用逐课通过制。请先独立完成练习题后再核对答案分支。

---
## 本课目标与完成标准

学完后你应能：
1. 掌握主流宽基指数（沪深300、中证500、标普500、MSCI）成分股定期调整（Index Rebalancing）的编制规则与时间表；
2. 深刻理解被动跟踪资金（Passive AUM / ETF / 指数增强基金）在调仓生效日的机械性交易行为与流动性挤兑效应；
3. 掌握基于市场微观结构的平方根市场冲击定律（Square-Root Law of Market Impact）与“日均成交量吸收天数（Days of ADV）”度量模型；
4. 编写代码实现指数调仓被动流动性冲击测算引擎，筛选高弹性套利标的，并评估开仓抢跑与收盘倒手（MOC）的期望收益。

通过标准：
- 独立补齐实战代码中的 TODO 填空，并通过全部 assert 单元测试；
- 深入回答 Q1~Q3 思考题，因果链完整，阐述跨期套利拥挤度与生效日“尾盘集合竞价流动性反转”现象。

---
## 核心心智模型与数理推导

### 1. 指数调整套利的生命周期与时间拓扑

```
        考察截止日                    官方公告日                    生效实施日 (T_eff)
            │                            │                               │
            ▼                            ▼                               ▼
    ┌───────────────┐            ┌───────────────┐               ┌───────────────┐
    │ 步骤 1：名单预测│───────────►│ 步骤 2：抢跑建仓│──────────────►│ 步骤 3：尾盘交割│
    └───────────────┘            └───────────────┘               └───────────────┘
     量化团队依据公开规则          公募与散户跟风买入               被动 ETF 必须在收盘
     提前 1~2 个月推算名单        推动调入股继续上涨               集合竞价不计成本无脑买齐;
     低位悄悄建立多头底仓         调出股承压下跌                  套利者高位精准倒手了结！
                                                                         │
                                                                         ▼
                                                                 ┌───────────────┐
                                                                 │ 步骤 4：均值反转│
                                                                 └───────────────┘
                                                                  人工流动性买盘消失，
                                                                  调入股大概率回落超跌
```

### 2. 被动资金流动性冲击模型 (Market Impact)

假设跟踪目标指数的被动资金资产总规模为 $\text{AUM}_{\text{passive}}$：
- **调入股票 $i$ 的目标被动买入金额**：
  $$V_{\text{buy}, i} = \text{AUM}_{\text{passive}} \times w_i$$
  其中 $w_i$ 为该股票在新指数中的理论权重。
- **日均成交金额吸收天数（Days of ADV）**：
  $$\text{Days of ADV}_i = \frac{V_{\text{buy}, i}}{\text{ADV}_i}$$
  其中 $\text{ADV}_i$ 为该股票近 20~60 个交易日的日均成交额（Average Daily Dollar Volume）。
  - 若 $\text{Days of ADV} \ge 3.0$：说明被动资金需要吃掉该股整整 3 天以上的全部市场交易量，必然形成显著的供需失衡与价格上推冲击！

- **平方根市场冲击定律（Square-Root Law）**：
  根据 Barra 与 Almgren-Chriss 微观结构理论，大额被动资金强行买入引发的相对价格冲击百分比与订单规模占 ADV 比例的平方根成正比：
  $$\text{Impact Pct} = \text{Sign} \times \eta \times \sqrt{\text{Days of ADV}}$$
  - 调入股（INCLUSION）：$\text{Sign} = +1$
  - 调出股（EXCLUSION）：$\text{Sign} = -1$
  - $\eta$ 为市场流动性冲击弹性系数（A股通常在 0.02~0.04 之间）。
"""

import math
from typing import List, Dict, Any


class IndexRebalanceArbitrageEngine:
    """
    指数成分股调整被动资金流动性冲击测算与套利分析引擎。
    """

    def __init__(self, passive_aum: float, impact_coefficient: float = 0.03):
        """
        :param passive_aum: 跟踪该指数的被动资金总规模 (美元或元)
        :param impact_coefficient: 平方根市场冲击弹性系数 eta (默认 0.03)
        """
        self.passive_aum = passive_aum
        self.impact_coefficient = impact_coefficient

    def estimate_liquidity_impact(self, stock_records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        测算每只调入/调出股票的被动资金流动性冲击。
        
        输入记录格式：
        stock_records = [
            {
                "symbol": "600100",
                "name": "A公司",
                "action": "INCLUSION",  # INCLUSION (调入) 或 EXCLUSION (调出)
                "index_weight": 0.015,  # 指数目标权重
                "adv_dollar": 30000000.0, # 近期日均成交金额
                "current_price": 20.0   # 消息前基准市价
            },
            ...
        ]
        """
        # TODO:
        # 1. 计算 passive_flow = passive_aum * index_weight
        # 2. 计算 days_of_adv = passive_flow / adv_dollar
        # 3. 计算 expected_impact_pct = sign * impact_coefficient * sqrt(days_of_adv)
        # 4. 计算 expected_target_price = current_price * (1 + expected_impact_pct)
        # 5. 返回结构化指标字典列表
        pass

    def screen_high_impact_arbitrage_targets(
        self,
        stock_records: List[Dict[str, Any]],
        min_days_of_adv: float = 3.0
    ) -> List[Dict[str, Any]]:
        """
        筛选具备高确定性流动性挤兑的重点套利标的 (Days of ADV >= min_days_of_adv)。
        按 days_of_adv 降序排列。
        """
        # TODO: 过滤 days_of_adv >= min_days_of_adv 并按降序排序返回
        pass


# ==============================================================================
# Q1~Q3 深度思考题（请在作答区填写因果链推导）
# ==============================================================================
r"""
### Q1：为什么随着量化对冲基金普遍建立“指数调整预测模型”，提前数周甚至数月抢跑（Front-running），导致指数调整官方公布日反而经常出现“利好兑现成利空（Sell the News）”的诡异暴跌？

**你的答案：**
[在此处写下你的分析：需阐述编制规则公开透明度导致的提前超额博弈，以及官宣日抢跑获利盘集中向次级追高资金倾销导致的流动性反噬]


### Q2：在生效日（Implementation Date）收盘集合竞价（MOC - Market-on-Close）阶段，为什么被动 ETF 基金经理即便明知股价被严重拉高，也必须不计成本挂涨停价甚至市价扫盘买齐？

**你的答案：**
[在此处写下你的分析：需从被动 ETF 的核心考核指标“跟踪误差（Tracking Error）最小化”出发，分析官方指数基准价格与盘中执行价格错配的不可逆风险]


### Q3：比较中国 A 股（如沪深 300 / 中证 500）与全球跨市场指数（如 MSCI 新兴市场指数）在成分股调整执行机制与流动性冲击上的本质区别。

**你的答案：**
[在此处写下你的分析：需对比境内单一货币结算 vs 跨境资金离岸兑换及北向资金黄金三分钟尾盘脉冲效应与反转交易机会]
"""


# ==============================================================================
# 单元测试与断言校验（请勿修改）
# ==============================================================================
if __name__ == "__main__":
    print("[Testing] IndexRebalanceArbitrageEngine Liquidity Impact Calculation...")
    engine = IndexRebalanceArbitrageEngine(passive_aum=10000000000.0, impact_coefficient=0.03)

    test_stocks = [
        {
            "symbol": "600100",
            "name": "A公司",
            "action": "INCLUSION",
            "index_weight": 0.015,
            "adv_dollar": 30000000.0,
            "current_price": 20.0
        },
        {
            "symbol": "000200",
            "name": "B公司",
            "action": "INCLUSION",
            "index_weight": 0.005,
            "adv_dollar": 50000000.0,
            "current_price": 15.0
        },
        {
            "symbol": "600300",
            "name": "C公司",
            "action": "EXCLUSION",
            "index_weight": 0.008,
            "adv_dollar": 20000000.0,
            "current_price": 10.0
        }
    ]

    impacts = engine.estimate_liquidity_impact(test_stocks)

    res_a = impacts[0]
    assert res_a["passive_flow"] == 150000000.0
    assert res_a["days_of_adv"] == 5.0
    assert res_a["expected_impact_pct"] == 6.71
    assert res_a["expected_target_price"] == 21.34

    res_b = impacts[1]
    assert res_b["days_of_adv"] == 1.0
    assert res_b["expected_impact_pct"] == 3.00
    assert res_b["expected_target_price"] == 15.45

    res_c = impacts[2]
    assert res_c["days_of_adv"] == 4.0
    assert res_c["expected_impact_pct"] == -6.00
    assert res_c["expected_target_price"] == 9.40

    print("[Testing] High Impact Arbitrage Screening...")
    screened = engine.screen_high_impact_arbitrage_targets(test_stocks, min_days_of_adv=3.0)
    assert len(screened) == 2
    assert screened[0]["symbol"] == "600100"
    assert screened[1]["symbol"] == "600300"
    assert screened[0]["days_of_adv"] > screened[1]["days_of_adv"]

    print("All assertions passed! Module 09 Lesson 03 verification successful.")

